# PEI Quick-Fix: Platt Scaling + Response Filter + Multiplicative PEI

This notebook applies three targeted improvements to the Phase 1 PEI pipeline:

1. **Platt scaling** on existing probes (fixes bimodal ISD artefact)
2. **Minimum response-length filter** (>=20 tokens, fixes HellaSwag single-letter problem)
3. **Multiplicative PEI** reported alongside additive (addresses unjustified composition choice)

**No GPU needed.** Everything runs on CPU using your saved `activations.npz` and `judged.jsonl`.

## Step 0: Mount Drive and find the project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import numpy as np
from pathlib import Path
from dataclasses import asdict

# Direct imports: flat repo structure
from probe_internals import load_activations, ProbeResult
from identify_errors import load_judged

results_path = Path(results_dir)
figures_path = results_path  # figures live in the same folder

# Load activations
print("Loading activations...")
activations = load_activations(results_path / 'activations.npz')
print(f"  Layers: {sorted(activations.keys())}")
for k, v in activations.items():
    print(f"  Layer {k}: shape {v.shape}")

# Load judged responses
print("\nLoading judged responses...")
judged = load_judged(results_path / 'judged.jsonl')
labels = np.array([1 if j.is_correct else 0 for j in judged])
n_errors = sum(labels == 0)
print(f"  Total responses: {len(judged)}")
print(f"  Errors: {n_errors} ({100 * n_errors / len(judged):.1f}%)")
print(f"  Correct: {sum(labels == 1)}")

In [ ]:
# Verify the project structure
if PROJECT_ROOT:
    print("Project contents:")
    for item in sorted(os.listdir(PROJECT_ROOT)):
        print(f"  {item}")
    print(f"\nResults dir contents:")
    if os.path.exists(results_dir):
        for item in sorted(os.listdir(results_dir)):
            size = os.path.getsize(os.path.join(results_dir, item))
            print(f"  {item} ({size / 1024:.0f} KB)")
    else:
        print("  Results directory not found!")
else:
    print("Set PROJECT_ROOT manually below:")
    # PROJECT_ROOT = '/content/drive/MyDrive/YOUR_PATH_HERE'
    # results_dir = os.path.join(PROJECT_ROOT, 'data', 'results')

In [ ]:
# Confirm we have what we need
import sys

assert PROJECT_ROOT is not None, "Set PROJECT_ROOT manually in the cell above"
assert os.path.exists(os.path.join(results_dir, 'activations.npz')), \
    f"activations.npz not found in {results_dir}"
assert os.path.exists(os.path.join(results_dir, 'judged.jsonl')), \
    f"judged.jsonl not found in {results_dir}"

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")
print("All required files present. Ready to proceed.")

## Step 1: Install dependencies

In [ ]:
!pip install -q scikit-learn numpy spacy matplotlib seaborn pandas scipy pyyaml tqdm
!python -m spacy download en_core_web_sm -q

In [ ]:
import os, sys

# Set these to your own paths
github_path = '/content/drive/MyDrive/YOUR_PATH/pei'  # repo root
results_path = '/content/drive/MyDrive/YOUR_PATH/pei_results'  # pipeline outputs

print("=== Project ===")
if os.path.exists(github_path):
    print(os.listdir(github_path))
    if os.path.exists(os.path.join(github_path, "src")):
        print("src/ found!")
        print(os.listdir(os.path.join(github_path, "src")))
else:
    print("Not found")

print("\n=== Results ===")
if os.path.exists(results_path):
    for f in os.listdir(results_path):
        size = os.path.getsize(os.path.join(results_path, f))
        print(f"  {f} ({size / 1024:.0f} KB)")
else:
    print("Not found")


In [ ]:
import os, sys

# Set these to match the paths above
PROJECT_ROOT = '/content/drive/MyDrive/YOUR_PATH/pei'
results_dir = '/content/drive/MyDrive/YOUR_PATH/pei_results'
figures_dir = results_dir  # figures are in the same directory

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

print(f"Working dir: {os.getcwd()}")


## Step 2: Load existing data

In [ ]:
import json
import numpy as np
from pathlib import Path
from dataclasses import asdict

from probe_internals import load_activations, ProbeResult
from identify_errors import load_judged

results_path = Path(results_dir)
figures_path = results_path / 'figures'
figures_path.mkdir(parents=True, exist_ok=True)

# Load activations
print("Loading activations...")
activations = load_activations(results_path / 'activations.npz')
print(f"  Layers: {sorted(activations.keys())}")
for k, v in activations.items():
    print(f"  Layer {k}: shape {v.shape}")

# Load judged responses
print("\nLoading judged responses...")
judged = load_judged(results_path / 'judged.jsonl')
labels = np.array([1 if j.is_correct else 0 for j in judged])
n_errors = sum(labels == 0)
print(f"  Total responses: {len(judged)}")
print(f"  Errors: {n_errors} ({100 * n_errors / len(judged):.1f}%)")
print(f"  Correct: {sum(labels == 1)}")

## Step 3: Re-train probes WITH Platt scaling

This is the key improvement. Instead of raw logistic regression probabilities (which cluster near 0 and 1, creating the bimodal ISD artefact), we apply Platt scaling on a held-out calibration set.

In [ ]:
import logging
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('pei')


def train_calibrated_probes(
    activations: dict,
    labels: np.ndarray,
    test_split: float = 0.2,
    random_seed: int = 42,
):
    """
    Train logistic regression probes with Platt scaling.

    Splits data into:
      - 68% probe training
      - 12% calibration (for Platt scaling)
      - 20% test (untouched, for evaluation)
    """
    probes = {}
    results = []

    # Same test split as Phase 1 for comparability
    X_train_idx, X_test_idx = train_test_split(
        np.arange(len(labels)), test_size=test_split,
        random_state=random_seed, stratify=labels,
    )

    for layer_idx, X in sorted(activations.items()):
        X_train_all, X_test = X[X_train_idx], X[X_test_idx]
        y_train_all, y_test = labels[X_train_idx], labels[X_test_idx]

        # Standardise: fit on train only
        scaler = StandardScaler()
        X_train_all = scaler.fit_transform(X_train_all)
        X_test = scaler.transform(X_test)

        # Split training into probe + calibration
        X_probe, X_cal, y_probe, y_cal = train_test_split(
            X_train_all, y_train_all, test_size=0.15,
            random_state=random_seed, stratify=y_train_all,
        )

        # Train base probe
        base_probe = LogisticRegression(
            max_iter=2000, solver='lbfgs', C=1.0, random_state=random_seed,
        )
        base_probe.fit(X_probe, y_probe)

        # Apply Platt scaling
        calibrated_probe = CalibratedClassifierCV(
            base_probe, method='sigmoid', cv='prefit',
        )
        calibrated_probe.fit(X_cal, y_cal)

        # Evaluate on held-out test set
        y_pred = calibrated_probe.predict(X_test)
        y_prob = calibrated_probe.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, y_pred)
        try:
            auc = roc_auc_score(y_test, y_prob)
        except ValueError:
            auc = 0.5

        probes[layer_idx] = (calibrated_probe, scaler)
        results.append(ProbeResult(
            layer_idx=layer_idx, accuracy=acc, auc=auc,
            n_train=len(X_probe), n_test=len(X_test),
        ))
        print(f"  Layer {layer_idx}: accuracy={acc:.3f}, AUC={auc:.3f} "
              f"(train={len(X_probe)}, cal={len(X_cal)}, test={len(X_test)})")

    return probes, results


print("Training calibrated probes...")
probes, probe_results = train_calibrated_probes(activations, labels)
print("\nDone.")

In [ ]:
# Save probes and results
import pickle

with open(results_path / 'probes.pkl', 'wb') as f:
    pickle.dump(probes, f)
print(f"Saved calibrated probes to {results_path / 'probes.pkl'}")

with open(results_path / 'probe_results.json', 'w') as f:
    json.dump([asdict(r) for r in probe_results], f, indent=2)
print(f"Saved probe results to {results_path / 'probe_results.json'}")

## Step 4: Compute calibrated ISD scores

In [ ]:
from probe_internals import ISDScore, save_isd_scores

def compute_calibrated_isd(activations, probes, judged, labels):
    """Compute ISD using calibrated probes."""
    scores = []

    for i in range(len(judged)):
        layer_scores = {}
        for layer_idx, (probe, scaler) in probes.items():
            x = activations[layer_idx][i].reshape(1, -1)
            x = scaler.transform(x)
            p_correct = probe.predict_proba(x)[0, 1]
            layer_scores[layer_idx] = float(p_correct)

        isd = float(np.mean(list(layer_scores.values())))

        scores.append(ISDScore(
            task_id=judged[i].task_id,
            domain=judged[i].domain,
            is_correct=bool(labels[i]),
            layer_scores=layer_scores,
            isd_score=isd,
        ))

    return scores


print("Computing calibrated ISD scores...")
isd_scores = compute_calibrated_isd(activations, probes, judged, labels)
save_isd_scores(isd_scores, results_path / 'isd_scores.jsonl')

# Quick check: distribution
error_isds = [s.isd_score for s in isd_scores if not s.is_correct]
correct_isds = [s.isd_score for s in isd_scores if s.is_correct]
print(f"\nCalibrated ISD distribution:")
print(f"  Correct:   mean={np.mean(correct_isds):.3f}, std={np.std(correct_isds):.3f}")
print(f"  Errors:    mean={np.mean(error_isds):.3f}, std={np.std(error_isds):.3f}")
print(f"  Separation: {np.mean(correct_isds) - np.mean(error_isds):.3f}")
print(f"  (Phase 1 was: correct=0.852, errors=0.232, separation=0.620)")

In [ ]:
# Plot ISD distributions: Phase 1 vs calibrated
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Calibrated ISD for errors
axes[0].hist(error_isds, bins=30, edgecolor='white', alpha=0.8, color='#DD8452')
axes[0].set_xlabel('ISD Score (calibrated)')
axes[0].set_ylabel('Count')
axes[0].set_title('Calibrated ISD Distribution - Errors Only')
axes[0].axvline(np.mean(error_isds), color='red', linestyle='--', label=f'Mean={np.mean(error_isds):.3f}')
axes[0].legend()

# Calibrated ISD: correct vs errors
axes[1].hist(correct_isds, bins=30, alpha=0.6, label=f'Correct (n={len(correct_isds)})', color='#55A868')
axes[1].hist(error_isds, bins=30, alpha=0.6, label=f'Errors (n={len(error_isds)})', color='#DD8452')
axes[1].set_xlabel('ISD Score (calibrated)')
axes[1].set_ylabel('Count')
axes[1].set_title('Calibrated ISD: Correct vs Errors')
axes[1].legend()

plt.tight_layout()
plt.savefig(figures_path / 'isd_calibrated_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nIs this less bimodal than Phase 1?")
print("If yes - the calibration fixed the artefact.")
print("If still bimodal - property of the probe not an artefact.")

## Step 5: Re-extract LCS (or load existing)

In [ ]:
from linguistic_features import LinguisticFeatures

lcs_path = results_path / 'linguistic_features.jsonl'

if lcs_path.exists():
    print("Loading existing LCS features...")
    lcs_features = []
    with open(lcs_path) as f:
        for line in f:
            d = json.loads(line)
            lcs_features.append(LinguisticFeatures(**d))
    print(f"  Loaded {len(lcs_features)} feature vectors")
else:
    print("LCS features not found: re-extracting (CPU, ~10 min)...")
    from linguistic_features import extract_all_features, save_features

    lcs_features = extract_all_features(
        task_ids=[j.task_id for j in judged],
        domains=[j.domain for j in judged],
        responses=[j.response for j in judged],
        spacy_model='en_core_web_sm',
    )
    save_features(lcs_features, lcs_path)
    print(f"  Extracted and saved {len(lcs_features)} feature vectors")

## Step 6: Compute PEI with response filter + multiplicative scoring

In [ ]:
from pei_score import normalise_scores, PEIResult, save_pei

MIN_RESPONSE_TOKENS = 20

# Build lookups
isd_lookup = {s.task_id: s for s in isd_scores}
lcs_lookup = {f.task_id: f for f in lcs_features}
response_lookup = {j.task_id: j.response for j in judged}

# Common IDs (errors only)
common_ids = set(isd_lookup.keys()) & set(lcs_lookup.keys())
common_ids = {tid for tid in common_ids if not isd_lookup[tid].is_correct}
print(f"Total errors before filter: {len(common_ids)}")

# Apply response-length filter
filtered_ids = {
    tid for tid in common_ids
    if len(response_lookup.get(tid, '').split()) >= MIN_RESPONSE_TOKENS
}
n_excluded = len(common_ids) - len(filtered_ids)
print(f"Excluded by length filter (<{MIN_RESPONSE_TOKENS} tokens): {n_excluded}")

# Check which domains lost responses
excluded_ids = common_ids - filtered_ids
excluded_domains = [isd_lookup[tid].domain for tid in excluded_ids]
from collections import Counter
print(f"Excluded by domain: {dict(Counter(excluded_domains))}")

task_ids = sorted(filtered_ids)
print(f"\nErrors after filter: {len(task_ids)}")

In [ ]:
# Extract raw scores and normalise
raw_isd = [isd_lookup[tid].isd_score for tid in task_ids]
raw_lcs = [lcs_lookup[tid].lcs_score for tid in task_ids]

norm_isd = normalise_scores(raw_isd, 'min_max')
norm_lcs = normalise_scores(raw_lcs, 'min_max')

# Compute both PEI compositions
pei_results = []
for i, tid in enumerate(task_ids):
    pei_add = 0.5 * norm_isd[i] + 0.5 * norm_lcs[i]
    pei_mult = norm_isd[i] * norm_lcs[i]
    pei_results.append(PEIResult(
        task_id=tid,
        domain=isd_lookup[tid].domain,
        is_correct=isd_lookup[tid].is_correct,
        isd_score=norm_isd[i],
        lcs_score=norm_lcs[i],
        pei_score=float(pei_add),
    ))

# Sort by additive PEI and assign ranks
pei_results.sort(key=lambda r: r.pei_score, reverse=True)
for rank, r in enumerate(pei_results, 1):
    r.pei_rank = rank

# We need to store multiplicative PEI alongside
# Since the dataclass may not have the field, store separately
pei_mult_scores = {
    tid: norm_isd[i] * norm_lcs[i]
    for i, tid in enumerate(task_ids)
}

# Save PEI results (additive)
save_pei(pei_results, results_path / 'pei_scores.jsonl')

# Also save an extended version with both compositions
with open(results_path / 'pei_scores_extended.jsonl', 'w') as f:
    for r in pei_results:
        d = asdict(r)
        d['pei_multiplicative'] = pei_mult_scores[r.task_id]
        f.write(json.dumps(d) + '\n')

pei_add_vals = [r.pei_score for r in pei_results]
pei_mult_vals = [pei_mult_scores[r.task_id] for r in pei_results]

print(f"Additive PEI:       mean={np.mean(pei_add_vals):.3f}, std={np.std(pei_add_vals):.3f}")
print(f"Multiplicative PEI: mean={np.mean(pei_mult_vals):.3f}, std={np.std(pei_mult_vals):.3f}")
print(f"Rank correlation (additive vs multiplicative): "
      f"{np.corrcoef(pei_add_vals, pei_mult_vals)[0, 1]:.3f}")

## Step 7: Regenerate all figures

In [ ]:
import pandas as pd
from scipy import stats

COLOURS = {'factual_qa': '#4C72B0', 'reasoning': '#DD8452', 'commonsense': '#55A868'}

df = pd.DataFrame([asdict(r) for r in pei_results])
df['pei_multiplicative'] = df['task_id'].map(pei_mult_scores)

print(f"DataFrame shape: {df.shape}")
print(f"\nPer-domain stats:")
for domain in df['domain'].unique():
    subset = df[df['domain'] == domain]
    print(f"  {domain}: n={len(subset)}, "
          f"PEI_add={subset['pei_score'].mean():.3f}, "
          f"PEI_mult={subset['pei_multiplicative'].mean():.3f}")

In [ ]:
# Figure 1: PEI distribution (additive)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(df['pei_score'], bins=30, edgecolor='white', alpha=0.8, color='#4C72B0')
axes[0].set_xlabel('PEI Score (Additive)')
axes[0].set_ylabel('Count')
axes[0].set_title('PEI Distribution (All Errors, Calibrated)')
axes[0].axvline(df['pei_score'].mean(), color='red', linestyle='--', label='Mean')
axes[0].legend()

for domain in df['domain'].unique():
    subset = df[df['domain'] == domain]
    axes[1].hist(subset['pei_score'], bins=20, alpha=0.6,
                 label=domain, color=COLOURS.get(domain))
axes[1].set_xlabel('PEI Score (Additive)')
axes[1].set_ylabel('Count')
axes[1].set_title('PEI Distribution by Domain')
axes[1].legend()

plt.tight_layout()
plt.savefig(figures_path / 'pei_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2: ISD vs LCS scatter
fig, ax = plt.subplots(figsize=(8, 6))

for domain in df['domain'].unique():
    subset = df[df['domain'] == domain]
    ax.scatter(subset['isd_score'], subset['lcs_score'],
              alpha=0.5, label=domain, color=COLOURS.get(domain), s=30)

ax.set_xlabel('ISD Score (calibrated)')
ax.set_ylabel('LCS Score')
ax.set_title('Internal-Surface Divergence vs Linguistic Confidence')
ax.legend()
ax.axhline(0.5, color='grey', linestyle=':', alpha=0.5)
ax.axvline(0.5, color='grey', linestyle=':', alpha=0.5)

r = df['isd_score'].corr(df['lcs_score'])
ax.text(0.75, 0.9, f'HIGH PEI\n(most dangerous)', transform=ax.transAxes,
        ha='center', va='center', fontsize=9, color='red', alpha=0.7)
ax.text(0.25, 0.1, f'LOW PEI\n(least dangerous)', transform=ax.transAxes,
        ha='center', va='center', fontsize=9, color='green', alpha=0.7)
ax.set_title(f'ISD vs LCS (r = {r:.3f})')

plt.tight_layout()
plt.savefig(figures_path / 'isd_vs_lcs.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"ISD-LCS correlation: r = {r:.3f} (Phase 1 was r = -0.13)")

In [ ]:
# Figure 3: Probe accuracy per layer
probe_df = pd.DataFrame([asdict(r) for r in probe_results])

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(probe_df))
width = 0.35

ax.bar(x - width/2, probe_df['accuracy'], width, label='Accuracy', color='#4C72B0')
ax.bar(x + width/2, probe_df['auc'], width, label='AUC', color='#DD8452')

ax.set_xlabel('Layer Index')
ax.set_ylabel('Score')
ax.set_title('Calibrated Probe Performance by Layer')
ax.set_xticks(x)
ax.set_xticklabels(probe_df['layer_idx'])
ax.legend()
ax.set_ylim(0.4, 1.0)
ax.axhline(0.5, color='grey', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig(figures_path / 'probe_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 4 (NEW): Additive vs Multiplicative PEI comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: additive vs multiplicative
axes[0].scatter(df['pei_score'], df['pei_multiplicative'], alpha=0.4, s=20, color='#4C72B0')
axes[0].set_xlabel('PEI (Additive)')
axes[0].set_ylabel('PEI (Multiplicative)')
axes[0].set_title('Additive vs Multiplicative PEI')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
r_comp = np.corrcoef(df['pei_score'], df['pei_multiplicative'])[0, 1]
axes[0].text(0.05, 0.95, f'r = {r_comp:.3f}', transform=axes[0].transAxes, va='top')

# Histogram of disagreements
disagreement = (df['pei_score'] - df['pei_multiplicative']).abs()
axes[1].hist(disagreement, bins=30, edgecolor='white', alpha=0.8, color='#DD8452')
axes[1].set_xlabel('|PEI_additive − PEI_multiplicative|')
axes[1].set_ylabel('Count')
axes[1].set_title('Composition Disagreement')
axes[1].axvline(disagreement.mean(), color='red', linestyle='--',
                label=f'Mean={disagreement.mean():.3f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(figures_path / 'pei_composition_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Top 20 disagreements
df['composition_disagreement'] = disagreement
top_disagreements = df.nlargest(20, 'composition_disagreement')
print("\nTop 20 composition disagreements (high on one dimension, low on the other):")
print(top_disagreements[['task_id', 'domain', 'isd_score', 'lcs_score',
                          'pei_score', 'pei_multiplicative', 'composition_disagreement'
                          ]].to_string(index=False))

## Step 8: Save summary statistics

In [ ]:
summary = {
    'n_errors_total': int(n_errors),
    'n_errors_after_filter': len(df),
    'n_excluded_by_filter': int(n_excluded),
    'min_response_tokens': MIN_RESPONSE_TOKENS,
    'probe_calibration': 'platt_scaling',
    'pei_additive_mean': float(df['pei_score'].mean()),
    'pei_additive_std': float(df['pei_score'].std()),
    'pei_additive_median': float(df['pei_score'].median()),
    'pei_multiplicative_mean': float(df['pei_multiplicative'].mean()),
    'pei_multiplicative_std': float(df['pei_multiplicative'].std()),
    'isd_lcs_correlation': float(df['isd_score'].corr(df['lcs_score'])),
    'composition_correlation': float(r_comp),
    'isd_correct_mean': float(np.mean(correct_isds)),
    'isd_error_mean': float(np.mean(error_isds)),
    'isd_separation': float(np.mean(correct_isds) - np.mean(error_isds)),
}

# Per-domain
for domain in df['domain'].unique():
    subset = df[df['domain'] == domain]
    summary[f'{domain}_n'] = len(subset)
    summary[f'{domain}_pei_additive_mean'] = float(subset['pei_score'].mean())
    summary[f'{domain}_pei_multiplicative_mean'] = float(subset['pei_multiplicative'].mean())

# Domain difference test
groups = [subset['pei_score'].values for _, subset in df.groupby('domain')]
if len(groups) > 1:
    stat, p = stats.kruskal(*groups)
    summary['domain_kruskal_h'] = float(stat)
    summary['domain_kruskal_p'] = float(p)

with open(results_path / 'summary_stats.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

In [ ]:
# Save example showcase
response_full_lookup = {j.task_id: j for j in judged}

def format_example(r):
    resp = response_full_lookup.get(r.task_id)
    if resp is None:
        return asdict(r)
    return {
        'task_id': r.task_id,
        'domain': r.domain,
        'pei_additive': round(r.pei_score, 3),
        'pei_multiplicative': round(pei_mult_scores.get(r.task_id, 0), 3),
        'isd_score': round(r.isd_score, 3),
        'lcs_score': round(r.lcs_score, 3),
        'prompt': resp.prompt[:200] + '...' if len(resp.prompt) > 200 else resp.prompt,
        'response': resp.response[:300] + '...' if len(resp.response) > 300 else resp.response,
        'ground_truth': resp.ground_truth,
        'extracted_answer': resp.extracted_answer,
    }

errors_only = [r for r in pei_results if not r.is_correct]
sorted_errors = sorted(errors_only, key=lambda r: r.pei_score, reverse=True)

examples = {
    'high_pei': [format_example(r) for r in sorted_errors[:5]],
    'low_pei': [format_example(r) for r in sorted_errors[-5:]],
}

with open(results_path / 'example_showcase.json', 'w') as f:
    json.dump(examples, f, indent=2)

print("=== TOP 5 HIGHEST-PEI ERRORS ===")
for ex in examples['high_pei']:
    print(f"\n  PEI={ex['pei_additive']:.3f} (mult={ex['pei_multiplicative']:.3f}), "
          f"ISD={ex['isd_score']:.3f}, LCS={ex['lcs_score']:.3f}")
    print(f"  Domain: {ex['domain']}")
    print(f"  Truth: {ex['ground_truth']}")
    print(f"  Response: {ex['response'][:150]}...")

print("\n=== BOTTOM 5 LOWEST-PEI ERRORS ===")
for ex in examples['low_pei']:
    print(f"\n  PEI={ex['pei_additive']:.3f} (mult={ex['pei_multiplicative']:.3f}), "
          f"ISD={ex['isd_score']:.3f}, LCS={ex['lcs_score']:.3f}")
    print(f"  Domain: {ex['domain']}")
    print(f"  Truth: {ex['ground_truth']}")
    print(f"  Response: {ex['response'][:150]}...")

## Step 9: Summary: what changed

Compare these results to Phase 1:

| Metric | Phase 1 | Calibrated |
|--------|---------|------------|
| ISD correct mean | 0.852 | Check above |
| ISD error mean | 0.232 | Check above |
| ISD separation | 0.620 | Check above |
| ISD-LCS correlation | −0.13 | Check above |
| N errors | 1,389 | Check above (post-filter) |
| Best probe layer AUC | 0.718 (L21) | Check above |

Key questions:
1. Is the ISD distribution less bimodal? → Calibration working
2. How many responses did the length filter exclude? → HellaSwag cleanup
3. How correlated are additive and multiplicative PEI? → Composition analysis
4. Did probe accuracy change? → Expected: slight decrease (smaller training set)

In [ ]:
print("All done. Updated files saved to:")
print(f"  {results_path}/")
for f in sorted(results_path.iterdir()):
    if f.is_file():
        size = f.stat().st_size
        print(f"    {f.name} ({size / 1024:.0f} KB)")
print(f"\n  {figures_path}/")
for f in sorted(figures_path.iterdir()):
    if f.is_file():
        print(f"    {f.name}")
print("\nNext: tidy the repo README, commit, and submit.")